In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [4]:
df.shape

(50000, 2)

In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
df.drop_duplicates(inplace=True)

In [8]:
df.shape

(49582, 2)

## Data Pre- processing 
- steps we have to perform :-
  1. convert to lowercase
  2. remove URLs (https, http)
  3. " HTML
  4. " Punctuations (, ? . # )
  5. " stopwards (and , or )
  6. stemming
  7. encode sentiments
  8. vectorization (converting text into nums) (TF-IDF)

  #### 1. Converting into lowercase 

In [9]:
df["review"] = df["review"].str.lower()

In [10]:
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,"bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,i am a catholic taught in parochial elementary...,negative
49998,i'm going to have to disagree with the previou...,negative


#### 2. remove url's

- using regular expression library . it helps to detect patterns and replace that text with another and so on..many operations 

In [11]:
import re 

In [12]:
# example 

sample_text = "abc is the word , abc" # 3 convert abc into xyz
new_text = re.sub("abc" , "xyz" , sample_text)  #sub is substitue parameter 

print(new_text)

xyz is the word , xyz


In [13]:
def remove_urls(text):
    text = re.sub(r"http\S+" , "" , text)  # sub(pattern , repl , string ) \S non whitespace chars , \s white space 
    return text
df["review"] = df["review"].apply(remove_urls) 

### 3. remove punctuations 

In [14]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "" , text)  
    return text
df["review"] = df["review"].apply(remove_punctuations) 

In [15]:
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,i am a catholic taught in parochial elementary...,negative
49998,im going to have to disagree with the previous...,negative


###  4. remove html

In [16]:
def remove_html(text):
    text = re.sub(r"<.*?>" , "" , text)  
    return text
df["review"] = df["review"].apply(remove_html) 

In [17]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 5. removing Stopwordsand tokennizations 

- using lib NLTK (natural language toolkit)
and its important modules 
1. punkt - for tokenization

In [18]:
import nltk 

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [19]:
from nltk.tokenize import word_tokenize 
from nltk.corpus import stopwords

In [20]:
# word_tokenize is to read tokens 
# example 
ex_token_texts = word_tokenize(sample_text)

ex_token_texts

['abc', 'is', 'the', 'word', ',', 'abc']

In [21]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word , "")

    return text 

df["review"] = df["review"].apply(remove_stopwords)

In [22]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming - converting msg into thier base form 
- running -> run 
- played -> play

1. using lib Porterstemming

In [23]:
from nltk.stem import PorterStemmer

In [24]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

### 7. Encoding 

In [25]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [26]:
y = df["sentiment"]

In [27]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

In [28]:
df

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1
...,...,...
49995,thought th move dd rght good job t wsnt s cret...,1
49996,bd plot bd dlogu bd ctng dotc drectng nnoyng p...,0
49997,ctholc tught n prochl elentri school nun tught...,0
49998,im gog dgree previou comnt side mlt e secd rte...,0


### 8. Vectorization 

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features= 5000)
X = tf.fit_transform(df["review"])

In [30]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057140 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3538)	0.05515484681268887
  (0, 2868)	0.09361182809242374
  (0, 4940)	0.11467310366614668
  (0, 3002)	0.47200539890110405
  (0, 1275)	0.1357698919196183
  (0, 2289)	0.049500237517476293
  (0, 1933)	0.0791260308382
  (0, 3550)	0.0963974330192639
  (0, 1362)	0.06162489377343992
  (0, 1963)	0.061560444992697486
  (0, 219)	0.08588920995304898
  (0, 1620)	0.0738170550485134
  (0, 4369)	0.041994187696759305
  (0, 4171)	0.17799685402440263
  (0, 3693)	0.033532198172897175
  (0, 4737)	0.26798942924092045
  (0, 3805)	0.04427609784380831
  (0, 4769)	0.05877405881441711
  (0, 1739)	0.037520883911174724
  (0, 4497)	0.07614066339174266
  (0, 3857)	0.17537900435282314
  (0, 1630)	0.06142445471882175
  (0, 1862)	0.07433134577032253
  (0, 3329)	0.06406818508428483
  (0, 3332)	0.0844754682576354
  :	:
  (49581, 4890)	0.10682334916138103
  (49581, 1542)	0.17584072573791829

## Dataset and Dataloader

In [31]:
from sklearn.model_selection import train_test_split

X_train , X_test  , y_train , y_test = train_test_split( X , y , test_size = 0.2 , random_state=42 )

In [32]:
X_train.shape

(39665, 5000)

In [33]:
X_test.shape

(9917, 5000)

In [34]:
import torch
from torch.utils.data import TensorDataset , DataLoader 

In [35]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3248304 stored elements and shape (39665, 5000)>

In [36]:
# convert sparse matrix into ndarray
X_train = X_train.toarray()
X_test = X_test.toarray()

In [39]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float() , 
    torch.from_numpy(y_train.values).float()
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float() ,
    torch.from_numpy(y_test.values).float()
)

In [40]:
train_loader = DataLoader(train_set , shuffle=True , batch_size=64)
test_loader = DataLoader(test_set , shuffle=True , batch_size=64)

### Build our RNN

In [42]:
import torch.nn as nn 
import torch.optim as optim

In [53]:
class RNN(nn.Module):
    def __init__(self , input_size , hidden_size=128 , num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers =  num_layers

        #Rnn layer
        self.rnn = nn.RNN(input_size , hidden_size , num_layers , batch_first=True)

        # fully connectd layer
        self.fc = nn.Linear(hidden_size , 1)

        
    def forward(self , x):
       #optional-> shape(no. of layers , batch size , hidden size)
      h0 = torch.zeros(self.num_layers , x.size(0) , self.hidden_size)

      out, _ = self.rnn(x , h0)
      # 1st value = hidden state of all the timesteps -> (batch , seq_length , hidden size)
      # 2nd value = final hidden state of last timestep 

      out = self.fc(out[: , -1 , :])
      return out

In [54]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)  # add singleton direction

        outputs = model(Xb)  # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze())  # (batch_size,) => probability

        loss = criterion(outputs, yb)  # compute loss
        loss.backward()  # backprop
        optimizer.step()  # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")